# Week 1 · Exercises (classes + NumPy)

- **1–4 (classes):** Tuesday. **5–12 (NumPy):** Friday.
- Run the first cell to load the checker. Write on paper first, then type it in.
- For the NumPy exercises: **no `for` loops**. Use vectorized operations, axes, masks, and broadcasting. The point is to think in arrays.
- After a PASS, try one input of your own. These tests are stricter than Week 0's.

The solutions are in `solutions/week1_solutions.py`.

In [114]:
import numpy as np

def _t1():
    r = Rectangle(2, 3)
    return r.area() == 6 and r.perimeter() == 10 and repr(r) == "Rectangle(2, 3)" and Rectangle(5, 1).area() == 5

def _t2():
    s = Stack()
    if not (s.is_empty() and len(s) == 0 and s.pop() is None and s.peek() is None):
        return False
    s.push(1); s.push(2); s.push(3)
    ok = len(s) == 3 and s.peek() == 3 and s.pop() == 3 and s.pop() == 2 and len(s) == 1 and not s.is_empty()
    other = Stack()                      # two stacks must not share items
    return ok and len(other) == 0

def _t3():
    sq = Square(4)
    return isinstance(sq, Rectangle) and sq.area() == 16 and sq.perimeter() == 16 and repr(sq) == "Rectangle(4, 4)"

def _t4():
    m = RunningMean()
    if m.mean() is not None:
        return False
    for x in [2, 4, 9]:
        m.add(x)
    return m.n == 3 and abs(m.mean() - 5.0) < 1e-9

def _t7():
    X = np.array([[1.0, 10.0], [2.0, 20.0], [3.0, 60.0]])
    Z = standardize(X)
    return Z.shape == X.shape and np.allclose(Z.mean(axis=0), 0) and np.allclose(Z.std(axis=0), 1) and np.allclose(X, [[1, 10], [2, 20], [3, 60]])

def _t11():
    a = softmax(np.array([1.0, 2.0, 3.0]))
    b = softmax(np.array([1000.0, 1001.0, 1002.0]))   # must not overflow
    return np.allclose(a, [0.09003057, 0.24472847, 0.66524096]) and np.allclose(a, b) and np.isclose(a.sum(), 1)

TESTS = {
    1: _t1, 2: _t2, 3: _t3, 4: _t4,
    5: lambda: np.array_equal(make_grid(3), [[0, 1, 2], [3, 4, 5], [6, 7, 8]]) and make_grid(4).shape == (4, 4) and make_grid(4)[3, 2] == 14,
    6: lambda: np.allclose(column_means(np.array([[1, 2, 3], [3, 4, 5]])), [2, 3, 4]) and column_means(np.ones((5, 7))).shape == (7,),
    7: _t7,
    8: lambda: count_above(np.array([5, 1, 7, 3, 9]), 4) == 3 and count_above(np.array([[1, 8], [9, 2]]), 5) == 2 and count_above(np.array([1, 2]), 10) == 0,
    9: lambda: np.array_equal(row_argmax(np.array([[0.1, 0.7, 0.2], [0.9, 0.05, 0.05], [0.2, 0.3, 0.5]])), [1, 0, 2]),
    10: lambda: np.isclose(mse(np.array([3.0, 5.0, 2.0]), np.array([2.5, 5.0, 4.0])), 1.4166666666666667) and mse(np.zeros(4), np.zeros(4)) == 0,
    11: _t11,
    12: lambda: np.allclose(linear_predict(np.array([[1.0, 2.0], [3.0, 4.0], [0.0, 0.0]]), np.array([0.5, -1.0]), 2.0), [0.5, -0.5, 2.0]) and linear_predict(np.ones((10, 3)), np.ones(3), 0.0).shape == (10,),
}


def check(n):
    """Run the hidden tests for exercise n and say whether they pass."""
    try:
        ok = bool(TESTS[n]())
        print(f"Exercise {n}: " + ("PASS" if ok else "not right yet. A test case gives the wrong answer (re-read the description and try the edge cases)"))
    except NameError as e:
        print(f"Exercise {n}: run the cell with your code first ({e})")
    except TypeError as e:
        if "NoneType" in str(e):
            print(f"Exercise {n}: something returned None. Not written yet, or a missing `return`?")
        elif "takes no arguments" in str(e):
            print(f"Exercise {n}: your class has no __init__ yet, or its __init__ doesn't take these inputs ({e})")
        else:
            print(f"Exercise {n}: your code raised TypeError: {e}")
    except AttributeError as e:
        if "'NoneType'" in str(e):
            print(f"Exercise {n}: something returned None. Not written yet, or a missing `return`?")
        else:
            print(f"Exercise {n}: a method or attribute is missing: {e}. Check the spelling, `self.`, and super().__init__()")
    except Exception as e:
        print(f"Exercise {n}: your code raised {type(e).__name__}: {e}")

## 1. Rectangle
Write a class `Rectangle` with `__init__(self, width, height)` and the methods `area()` and `perimeter()`. Also write `__repr__` so that `repr(Rectangle(2, 3))` is exactly `"Rectangle(2, 3)"`.

In [115]:
class Rectangle:
    def __init__(self,width,height):
        self.width = width
        self.height = height

    def area(self):
        return self.width * self.height

    def perimeter(self):
        return 2*(self.width+self.height)

    def __repr__(self):
        return f"Rectangle{(self.width,self.height)}"

In [116]:
check(1)

Exercise 1: PASS


## 2. Stack
A **stack** is a pile where you can only add to the top or take from the top. Write `Stack` with:
- `push(x)`: put x on top
- `pop()`: remove and return the top item (or `None` if it's empty)
- `peek()`: return the top item **without** removing it (or `None`)
- `is_empty()`, and `__len__`

Hint: keep a list in `self.items`. **Create it in `__init__`**, not as a default argument, or two stacks would share one list (the Week 0 trap).

In [117]:
class Stack:
    def __init__(self):
        self.items = []

    def push(self,x):
        self.items.append(x) #if written as return self.items.append(x) there are a few bugs : 왜냐하면 function에서 append하는 행위를 return하는 것이기 때문. 
                             #1.self.items.append(x) runs, and x is added to the list. 2.append changes the list in place and returns None. It doesn’t give back the updated list. 3.return then sends that None back to whoever called push
    def pop(self):
        if not self.items:
            return None
        return self.items.pop()

    def peek(self):
        if not self.items:
            return None
        
        return self.items[-1]

    def is_empty(self):
        if not self.items:
            return True
        return False

    def __len__(self):
        return len(self.items)

In [118]:
check(2)

Exercise 2: PASS


## 3. Square inherits from Rectangle
Write `Square(Rectangle)` whose `__init__(self, side)` calls `super().__init__(side, side)`. Don't write `area`, `perimeter`, or `__repr__` again: they must come from `Rectangle`. Then `Square(4).area()` is 16 and `repr(Square(4))` is `"Rectangle(4, 4)"`.

In [119]:
class Square(Rectangle):
    def __init__(self,side):
        self.side = super().__init__(side,side)

In [120]:
check(3)

Exercise 3: PASS


## 4. RunningMean
Keep a running average $\bar x = \frac{1}{n}\sum_{i=1}^{n} x_i$ without storing every number. `__init__` sets `self.n = 0` and `self.total = 0`. `add(x)` updates both. `mean()` returns `total / n`, or `None` before anything has been added. (Training loops track the average loss exactly like this.)

In [121]:
class RunningMean:
    def __init__(self):
        self.n = 0
        self.total = 0

    def add(self,x):
        self.n +=  1
        self.total += x

    def mean(self):
        if self.n == 0:
            return None
        return self.total / self.n

In [122]:
check(4)

Exercise 4: PASS


## 5. Number grid
Return an n × n array whose entry at `[i, j]` is `i * n + j`. For example, `make_grid(3)` is `[[0, 1, 2], [3, 4, 5], [6, 7, 8]]`. One line: `arange` + `reshape`.

--> this just teaches us how .reshape in Numpy works?

In [123]:
def make_grid(n):
    return np.arange(n**2).reshape(n,n)

In [124]:
check(5)

Exercise 5: PASS


## 6. Column means
Return the mean of each **column** of a 2-D array, as shape `(number of columns,)`. Which axis?

In [125]:
def column_means(X):
    return X.mean(axis=0)

# axis=0 --> per column
# axis=1 --> per row

In [126]:
check(6)

Exercise 6: PASS


## 7. Standardize
Return a new array in which each column has mean 0 and standard deviation 1: $Z_{ij} = \frac{X_{ij} - \mu_j}{\sigma_j}$, with $\mu_j$ and $\sigma_j$ the mean and standard deviation of column $j$. Use broadcasting, and don't change `X` itself.

In [127]:
def standardize(X):
    return (X - X.mean(axis=0))/X.std(axis=0)

In [128]:
check(7)

Exercise 7: PASS


## 8. Count above a threshold
Return how many elements of `a` (any shape) are greater than `t`. Use a mask and `.sum()`.

In [129]:
def count_above(a, t):
    return np.where(a.reshape(-1) > t,1,0).sum()

# Below is a more efficient answer. We do not need reshape and np.where
#def count_above(a, t):
#    return (a > t).sum()

#a > t gives a boolean array of the same shape as a, with True where the element is above t.
#.sum() with no axis adds up every element across all axes, counting each True as 1. That’s why no reshape is needed for “any shape.”

In [130]:
check(8)

Exercise 8: PASS


## 9. Predicted class per row
Each row holds a classifier's scores for one sample. Return the index of the highest score in each row: `[[0.1, 0.7, 0.2], ...]` gives `[1, ...]`.

In [131]:
def row_argmax(scores):
    return scores.argmax(axis=1) 

In [132]:
check(9)

Exercise 9: PASS


## 10. Mean squared error
Return $\text{MSE} = \frac{1}{n}\sum_{i=1}^{n}(y_i - \hat y_i)^2$, with no loop.

In [133]:
def mse(y, y_hat):
    return ((y - y_hat)**2).sum()/y.size

In [134]:
check(10)

Exercise 10: PASS


## 11. Softmax (the stable way)
$\operatorname{softmax}(z)_i = \dfrac{e^{z_i}}{\sum_j e^{z_j}}$ turns scores into probabilities that add up to 1. But $e^{1000}$ overflows to `inf`. The fix is to subtract $\max_j z_j$ first. Since $\frac{e^{z_i - c}}{\sum_j e^{z_j - c}} = \frac{e^{z_i}}{\sum_j e^{z_j}}$ for any constant $c$, the answer is identical, but the numbers stay small. Your version must work for `[1000, 1001, 1002]`.

In [135]:
def softmax(z):
    return np.exp(z - z.max())/np.exp(z - z.max()).sum()

In [136]:
check(11)

Exercise 11: PASS


## 12. Linear model prediction
$X \in \mathbb{R}^{n \times d}$ (shape `(n, d)`), $w \in \mathbb{R}^{d}$ (shape `(d,)`), and $b$ is a number. Return the predictions $\hat y = Xw + b$, shape `(n,)`. This one line is linear regression's prediction step.

In [137]:
def linear_predict(X, w, b):
    return X@w + b

In [138]:
check(12)

Exercise 12: PASS


---
### Run all checks

In [139]:
for n in [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]:
    check(n)

Exercise 1: PASS
Exercise 2: PASS
Exercise 3: PASS
Exercise 4: PASS
Exercise 5: PASS
Exercise 6: PASS
Exercise 7: PASS
Exercise 8: PASS
Exercise 9: PASS
Exercise 10: PASS
Exercise 11: PASS
Exercise 12: PASS


When all 12 pass, commit and push, then log your hours and which exercises needed hints.